## Feature Engineering

Based on EDA findings, we proceed to build the following features:
- Customer-product interaction matrix
- Customer behavioural features
- Product features

### Signal Strength Justification

We will assign the following weights to each interaction signal:

**Event weights (for interaction matrix):**
- `purchase`: 3.0 - strongest signal. Represents completed transactions from event tracking
- `cart`: 2.0 - strong intent signal. Customer considered the product seriously enough to add to cart
- `view`: 1.0 - weakest signal. Represents awareness only, no commitment

**Interaction score weights (for combined score):**
- `purchase_score`: 0.4 - ground truth purchase data from order records. Most reliable signal
- `purchase_count`: 0.3 - confirms purchase behaviour from event tracking
- `cart_count`: 0.2 - strong intent but no completion
- `view_count`: 0.1 - weakest behavioural signal

**Justification:**
The behavioral funnel (350k views to 100k carts to 50k purchases) confirms a 7:2:1 drop-off ratio across event types. Purchase events are 7x rarer than views, making them significantly more valuable as a signal. Weights reflect this natural funnel hierarchy.

Order records (purchase_score) are weighted highest because they come from transaction systems more reliable than event tracking which can miss or duplicate events.

### Import Libraries

In [1]:
import pandas as pd
import pyarrow
from sklearn.preprocessing import MinMaxScaler
import numpy as np

### Load the Dataset

In [2]:
customers = pd.read_parquet('../../../data/raw/customers.parquet')
orders = pd.read_parquet('../../../data/raw/orders.parquet')
products = pd.read_parquet('../../../data/raw/products.parquet')
events = pd.read_parquet('../../../data/raw/events.parquet')

### Section 1: Feature Engineering

#### A. Interaction Matrix - Order and Event Tables

In [3]:
# How much has each customer bought of each product?
order_interactions = (orders.groupby(['customer_id', 'product_id'])['quantity'].sum()).reset_index()

order_interactions = order_interactions.rename(columns={'quantity': 'purchase_score'})
order_interactions

,customer_id,product_id,purchase_score
0,CUST000001,PROD00137,3
1,CUST000001,PROD00485,4
2,CUST000001,PROD01665,4
3,CUST000001,PROD02074,2
4,CUST000001,PROD02818,1
...,...,...,...
499707,CUST100000,PROD01953,3
499708,CUST100000,PROD02232,2
499709,CUST100000,PROD02934,4
499710,CUST100000,PROD03602,5


Order interacttion shows 499,712 unique customer-product pairs with a purchase score between 1 and 10. Each row is evidence that a specific customer bought a specific product - and how much of it they bought.

In [4]:
order_interactions['purchase_score'].describe()

count    499712.000000
mean          2.999904
std           1.415600
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          10.000000
Name: purchase_score, dtype: float64

The purchase score ranging from 1–10 with a mean of 3 shows that customers have genuine and have varied purchase history across multiple products which will serve as foundation for Shopflow recommendation engine.
The tight distribution also shows that no single customer dominates the signal.

In [5]:
# To assign weights to event type based on value to Shopflow
# weight_map = {
    #'purchase': 3.0,
    #'cart': 2.0,
    #'view': 1.0
#}

# events['event_weight'] = events['event_type'].map(weight_map)

# How much has each customer bought of each product?
# event_interactions = events.groupby(['customer_id', 'product_id'])['event_weight'].sum().reset_index()

# event_interactions = event_interactions.rename(columns = {'event_weight': 'event_score'})
# event_interactions

In [6]:
# How many of these events each customer performed?
event_interactions = events.pivot_table(
    index=['customer_id', 'product_id'],
    columns='event_type',
    values='event_id',
    aggfunc='count'
).reset_index().fillna(0)

event_interactions.columns.name = None

# Rename columns for ease
event_interactions = event_interactions.rename(columns={
    'cart': 'cart_count', 
    'purchase': 'purchase_count', 
    'view': 'view_count'})

event_interactions

,customer_id,product_id,cart_count,purchase_count,view_count
0,CUST000001,PROD01323,1.0,0.0,0.0
1,CUST000001,PROD01748,0.0,0.0,1.0
2,CUST000001,PROD01753,0.0,0.0,1.0
3,CUST000001,PROD02612,0.0,0.0,1.0
4,CUST000001,PROD03559,0.0,0.0,1.0
...,...,...,...,...,...
499721,CUST099999,PROD04107,1.0,0.0,0.0
499722,CUST100000,PROD01489,1.0,0.0,0.0
499723,CUST100000,PROD03386,0.0,0.0,1.0
499724,CUST100000,PROD03796,0.0,0.0,1.0


This table captures customer interest beyond just purchases. It shows not only what customers bought, but what caught their attention, what they browsed, what they seriously considered by adding to cart, and what they ultimately bought. For ShopFlow, this means the recommendation engine can spot products a customer is interested in even before they make a purchase, helping recover some of that 68% cart abandonment rate.

In [7]:
event_interactions.describe()

,cart_count,purchase_count,view_count
count,499726.000000,499726.000000,499726.000000
mean,0.199409,0.100451,0.700688
std,0.399617,0.300647,0.458512
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,1.000000
75%,0.000000,0.000000,1.000000
max,2.000000,2.000000,2.000000


Most customers interact with a product in one way only - they either glance at it, add it to cart, or buy it, but rarely all three. This tells ShopFlow that the journey from awareness to purchase is often direct rather than gradual. A customer who views a product is unlikely to come back and cart it later - they either act immediately or move on.

In [8]:
# Merge both interactions together
interaction_matrix = pd.merge(order_interactions, event_interactions, on=['customer_id', 'product_id'], how='outer').fillna(0)

# scale the features for comparability
scaler = MinMaxScaler()

interaction_matrix_scaled = interaction_matrix.copy()
interaction_matrix_scaled[['purchase_score', 'cart_count', 'purchase_count', 'view_count']] = scaler.fit_transform(interaction_matrix[['purchase_score', 'cart_count', 'purchase_count', 'view_count']])

interaction_matrix_scaled


,customer_id,product_id,purchase_score,cart_count,purchase_count,view_count
0,CUST000001,PROD00137,0.3,0.0,0.0,0.0
1,CUST000001,PROD00485,0.4,0.0,0.0,0.0
2,CUST000001,PROD01665,0.4,0.0,0.0,0.0
3,CUST000001,PROD02074,0.2,0.0,0.0,0.0
4,CUST000001,PROD02818,0.1,0.0,0.0,0.0
...,...,...,...,...,...,...
998920,CUST099999,PROD04107,0.0,0.5,0.0,0.0
998921,CUST100000,PROD01489,0.0,0.5,0.0,0.0
998922,CUST100000,PROD03386,0.0,0.0,0.0,0.5
998923,CUST100000,PROD03796,0.0,0.0,0.0,0.5


Previously a purchase score of 10 would have drowned out a view count of 1 simply because of the difference in scale, not because it's necessarily 10 times more important has bben levelled between different types of customer behaviour to allow the model fairly evaluate all four signals together and learn from the patterns across all of them usinf minmaxscaler

In [9]:
# Time Decay
# Days since each order
orders['days_since_order'] = (pd.Timestamp.now() - orders['timestamp']).dt.days

# most recent interaction per customer-product pair
days_since_order = orders.groupby(['customer_id', 'product_id'])['days_since_order'].min().reset_index()

# apply decay formula
days_since_order['time_decay'] = np.exp(-0.001 * days_since_order['days_since_order'])

interaction_matrix_scaled = interaction_matrix_scaled.merge(
    days_since_order[['customer_id', 'product_id', 'time_decay']], 
    on=['customer_id', 'product_id'], 
    how='left'

)
interaction_matrix_scaled['time_decay'] = interaction_matrix_scaled['time_decay'].fillna(0)

interaction_matrix_scaled.head()

,customer_id,product_id,purchase_score,cart_count,purchase_count,view_count,time_decay
0,CUST000001,PROD00137,0.3,0.0,0.0,0.0,0.500574
1,CUST000001,PROD00485,0.4,0.0,0.0,0.0,0.404947
2,CUST000001,PROD01665,0.4,0.0,0.0,0.0,0.692117
3,CUST000001,PROD02074,0.2,0.0,0.0,0.0,0.459324
4,CUST000001,PROD02818,0.1,0.0,0.0,0.0,0.422317


In [10]:
# To sum up these scores by assigning weight
interaction_matrix_scaled['interaction_score'] = (
    interaction_matrix_scaled['purchase_score'] * 0.4 +
    interaction_matrix_scaled['purchase_count'] * 0.3 +
    interaction_matrix_scaled['cart_count'] * 0.2 +
    interaction_matrix_scaled['view_count'] * 0.1
) * interaction_matrix_scaled['time_decay']

interaction_matrix_scaled

,customer_id,product_id,purchase_score,cart_count,purchase_count,view_count,time_decay,interaction_score
0,CUST000001,PROD00137,0.3,0.0,0.0,0.0,0.500574,0.060069
1,CUST000001,PROD00485,0.4,0.0,0.0,0.0,0.404947,0.064791
2,CUST000001,PROD01665,0.4,0.0,0.0,0.0,0.692117,0.110739
3,CUST000001,PROD02074,0.2,0.0,0.0,0.0,0.459324,0.036746
4,CUST000001,PROD02818,0.1,0.0,0.0,0.0,0.422317,0.016893
...,...,...,...,...,...,...,...,...
998920,CUST099999,PROD04107,0.0,0.5,0.0,0.0,0.000000,0.000000
998921,CUST100000,PROD01489,0.0,0.5,0.0,0.0,0.000000,0.000000
998922,CUST100000,PROD03386,0.0,0.0,0.0,0.5,0.000000,0.000000
998923,CUST100000,PROD03796,0.0,0.0,0.0,0.5,0.000000,0.000000


In [11]:
interaction_matrix_scaled['interaction_score'].describe()

count    998925.000000
mean          0.028876
std           0.036187
min           0.000000
25%           0.000000
50%           0.012691
75%           0.054136
max           0.270823
Name: interaction_score, dtype: float64

The recommendation engine now prioritises what customers are interested in right now, not just what they bought historically. A customer who bought Sports equipment last week gets a stronger Sports recommendation signal than a customer who bought the same product two years ago,  even if the purchase quantities were identical.

#### B. Customer Features - Customer and Order Tables

In [12]:
# Days since signup to ascertain tenure which help determine their preferences
today_date = pd.Timestamp.now()
days_since_signup = (today_date - customers['signup_date']).dt.days

customers['tenure_days'] = days_since_signup

In [13]:
# Total Spend
customer_spend = orders.groupby('customer_id')['amount'].sum().rename('total_spend')

# To add to customer table
customers = customers.merge(customer_spend, on='customer_id', how='left')
customers['total_spend'] = customers['total_spend'].fillna(0)

In [14]:
# Average order Value per customer
avg_cust__order = orders.groupby('customer_id')['amount'].mean().rename('average_order')

# To add to customer table
customers = customers.merge(avg_cust__order, on='customer_id', how='left')
customers['average_order'] = customers['average_order'].fillna(0)

In [15]:
# Favourite category per customer
favorite = orders.merge(products[['product_id', 'category']], on='product_id', how='left')

customer_favorite = favorite.groupby(['customer_id', 'category']).size().groupby('customer_id').idxmax().map(lambda x: x[1]).rename('favorite_category')

# To add to customer table
customers = customers.merge(customer_favorite, on='customer_id', how='left')
customers['favorite_category'] = customers['favorite_category'].fillna('Unknown')

In [16]:
# how many orders a customer has placed
orders_placed = orders.groupby('customer_id')['order_id'].count().rename('orders_frequency')

# To add to customer table
customers = customers.merge(orders_placed, on='customer_id', how='left')
customers['orders_frequency'] = customers['orders_frequency'].fillna(0)

In [17]:
# how many different products they've bought
unique_products = orders.groupby('customer_id')['product_id'].nunique().rename('unique_products')

# To add to customer table
customers = customers.merge(unique_products, on='customer_id', how='left')
customers['unique_products'] = customers['unique_products'].fillna(0)

In [18]:
# Encode loyalty tier to prioritize more loyal customers
loyalty_map = {
    'Churned': 0,
    'Low Value': 1,
    'Medium Value': 2,
    'High Value': 3
}
	
customers['loyalty_tier'] = customers['loyalty_tier'].map(loyalty_map).fillna(0)

#### C. Product Features - Product and Order Tables

In [19]:
purchase_popularity = orders.groupby('product_id')['customer_id'].nunique().rename('purchase_popularity')

# To add to product table
products = products.merge(purchase_popularity, on='product_id', how='left')
products['purchase_popularity'] = products['purchase_popularity'].fillna(0)

In [20]:
# Average rating/score across all customers
average_interaction_score = interaction_matrix_scaled.groupby('product_id')['interaction_score'].mean().rename('average_rating')

# To add to product table
products = products.merge(average_interaction_score, on='product_id', how='left')
products['average_rating'] = products['average_rating'].fillna(0)

In [21]:
# Encode stock status to help deprioritise products with low stock status scores
stock_weight = {
    'Out of Stock':  0,
    'Low Stock': 1,
    "In Stock": 2
}
products['stock_status_encoded'] = products['stock_status'].map(stock_weight)

products.head()

,product_id,product_name,category,brand,price,cost,stock_status,purchase_popularity,average_rating,stock_status_encoded
0,PROD00001,Cultural Home,Home,BrandA,433.62,276.66,In Stock,89,0.024503,2
1,PROD00002,Unit Home,Home,BrandC,44.75,28.64,In Stock,101,0.033822,2
2,PROD00003,Send Sports,Sports,BrandC,442.87,227.93,In Stock,98,0.027925,2
3,PROD00004,Name Electronics,Electronics,Generic,319.51,138.49,In Stock,109,0.030956,2
4,PROD00005,What Books,Books,BrandC,141.91,58.26,In Stock,106,0.028409,2


In [22]:
# Encode categorical variables using onehotencoder
customers = pd.concat([customers, pd.get_dummies(customers['favorite_category'], prefix='cat')], axis=1)
products = pd.concat([products, pd.get_dummies(products['category'], prefix='cat')], axis=1)

# Drop original encoded columns
customers = customers.drop(columns=['favorite_category'])
products = products.drop(columns=['category'])

In [23]:
# Drop columns that are irrelevant to the objective (which product is this customer most likely to want next)
customers = customers.drop(columns=['email', 'first_name', 'last_name', 'city', 'country',
       'signup_date'])
products = products.drop(columns=['brand', 'price', 'cost', 'stock_status'])

In [24]:
# Rename customer category columns to avoid duplicate
customers = customers.rename(columns={
    'cat_Beauty': 'cust_Beauty',
    'cat_Books': 'cust_Books',
    'cat_Clothing': 'cust_Clothing',
    'cat_Electronics': 'cust_Electronics',
    'cat_Home': 'cust_Home',
    'cat_Sports': 'cust_Sports',
    'cat_Unknown': 'cust_Unknown'
})

products = products.rename(columns={
    'cat_Beauty': 'prod_Beauty',
    'cat_Books': 'prod_Books',
    'cat_Clothing': 'prod_Clothing',
    'cat_Electronics': 'prod_Electronics',
    'cat_Home': 'prod_Home',
    'cat_Sports': 'prod_Sports'
})

### Section 2: Merge all Engineered Features

In [25]:
interaction_matrix_scaled = interaction_matrix_scaled.merge(products, on='product_id', how='left')
rec_features = interaction_matrix_scaled.merge(customers, on='customer_id', how='left')

print(rec_features.shape)
rec_features.head()

(998925, 31)


,customer_id,product_id,purchase_score,cart_count,purchase_count,view_count,time_decay,interaction_score,product_name,purchase_popularity,...,average_order,orders_frequency,unique_products,cust_Beauty,cust_Books,cust_Clothing,cust_Electronics,cust_Home,cust_Sports,cust_Unknown
0,CUST000001,PROD00137,0.3,0.0,0.0,0.0,0.500574,0.060069,Fast Beauty,97,...,839.493333,6.0,6.0,True,False,False,False,False,False,False
1,CUST000001,PROD00485,0.4,0.0,0.0,0.0,0.404947,0.064791,Matter Sports,107,...,839.493333,6.0,6.0,True,False,False,False,False,False,False
2,CUST000001,PROD01665,0.4,0.0,0.0,0.0,0.692117,0.110739,Source Beauty,97,...,839.493333,6.0,6.0,True,False,False,False,False,False,False
3,CUST000001,PROD02074,0.2,0.0,0.0,0.0,0.459324,0.036746,Outside Electronics,107,...,839.493333,6.0,6.0,True,False,False,False,False,False,False
4,CUST000001,PROD02818,0.1,0.0,0.0,0.0,0.422317,0.016893,Get Sports,102,...,839.493333,6.0,6.0,True,False,False,False,False,False,False


In [26]:
# Convert bool to integers
rec_features = rec_features.replace({True: 1, False: 0})

rec_features.head()

,customer_id,product_id,purchase_score,cart_count,purchase_count,view_count,time_decay,interaction_score,product_name,purchase_popularity,...,average_order,orders_frequency,unique_products,cust_Beauty,cust_Books,cust_Clothing,cust_Electronics,cust_Home,cust_Sports,cust_Unknown
0,CUST000001,PROD00137,0.3,0.0,0.0,0.0,0.500574,0.060069,Fast Beauty,97,...,839.493333,6.0,6.0,1,0,0,0,0,0,0
1,CUST000001,PROD00485,0.4,0.0,0.0,0.0,0.404947,0.064791,Matter Sports,107,...,839.493333,6.0,6.0,1,0,0,0,0,0,0
2,CUST000001,PROD01665,0.4,0.0,0.0,0.0,0.692117,0.110739,Source Beauty,97,...,839.493333,6.0,6.0,1,0,0,0,0,0,0
3,CUST000001,PROD02074,0.2,0.0,0.0,0.0,0.459324,0.036746,Outside Electronics,107,...,839.493333,6.0,6.0,1,0,0,0,0,0,0
4,CUST000001,PROD02818,0.1,0.0,0.0,0.0,0.422317,0.016893,Get Sports,102,...,839.493333,6.0,6.0,1,0,0,0,0,0,0


### Section 3: Save the Features

In [27]:
rec_features.to_parquet('../../../data/processed/rec_features.parquet', index=False)

Index(['product_id', 'product_name', 'purchase_popularity', 'average_rating',
       'stock_status_encoded', 'prod_Beauty', 'prod_Books', 'prod_Clothing',
       'prod_Electronics', 'prod_Home', 'prod_Sports'],
      dtype='object')

## Feature Engineering Summary & Modelling Handoff

### What was built

**Interaction Matrix**
- Combined purchase signals from orders and behavioural signals from events
- 998,925 unique customer-product pairs
- Four interaction signals: purchase_score, cart_count, purchase_count, view_count
- All signals MinMax scaled to 0–1 for comparability
- Time decay applied using formula: e^(−0.001 × days_since_most_recent_order)
  per customer-product pair — recent interactions weighted higher than older ones
- Single combined interaction_score computed using weighted formula:
  (purchase_score × 0.4 + purchase_count × 0.3 + cart_count × 0.2 + view_count × 0.1) × time_decay

**Customer Features**
- tenure_days, total_spend, average_order, orders_frequency, unique_products
- loyalty_tier encoded ordinally (0–3)
- favourite_category one-hot encoded into 7 binary columns (cust_ prefix)

**Product Features**
- purchase_popularity, average_rating
- stock_status encoded ordinally (0–2) to deprioritise unavailable products
- category one-hot encoded into 6 binary columns (prod_ prefix)

**Final feature set**
- rec_features.parquet — 998,925 rows × 31 columns
- Committed to data_science/feature_store/
- Primary key: customer_id + product_id


### Modelling Inputs

The following will feed directly into the recommendation model:

- `interaction_score` — primary signal for collaborative filtering,
   enriched with time decay
- `time_decay` — recency signal, confirms how fresh each interaction is
- Customer features — for finding similar customers
- Product features — for finding similar products
- `stock_status_encoded` — to filter out unavailable recommendations


### Next Step — Modelling

With rec_features.parquet committed, we proceed to:
1. Build the customer-product matrix for collaborative filtering
2. Train item-item similarity model
3. Generate top-5 product recommendations per customer
4. Evaluate using relevance score, precision@k and recall@k
5. Register best model in MLflow